In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/ezzeddinesayed52@gmail.com/FMCG-DE_Project/Scripts/Setups/utilities

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","products","Data Source")


In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
base_path = (
    f"r2://sportsbar@"
    f"eefc59d684bff864c9de01ed688857a0"
    f".r2.cloudflarestorage.com/"
    f"{data_source}/*.csv"
)
base_path

In [0]:
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*","_metadata.file_name","_metadata.file_size")
)
df.display(10)

In [0]:
df.printSchema()

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
%sql
SELECT *
FROM fmcg.bronze.products
LIMIT 10

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
display(df_bronze.limit(10))

In [0]:
df_bronze.select("category").distinct().show()
df_bronze.select("product_name").distinct().show()

In [0]:
df_silver = (
    df_bronze.withColumn(
        "variant", F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1)
    )
    .withColumn(
        "product_name",
        F.trim(F.regexp_replace(F.col("product_name"), r"\s*\(.*?\)", "")),
    )
)
display(df_silver.limit(5))

In [0]:
df_silver.groupBy(df_silver.columns).count().filter(F.col("count") > 1).count()

In [0]:
df_silver = df_silver.dropDuplicates()
df_silver.groupBy(df_silver.columns).count().filter(F.col("count") > 1).count()

In [0]:
df_silver = (
    df_silver.withColumn(
        "category",F.initcap("category")
    )
)
df_silver.display(5)

In [0]:
#  Fix typo 
df_silver = (
    df_silver.withColumn(
        "product_name",
        F.regexp_replace(F.col("product_name"), "(?i)Protien", "Protein"),
    )
    .withColumn(
        "category",
        F.regexp_replace(F.col("category"), "(?i)Protien", "Protein"),
    )
)
df_silver.show(10, truncate=False)

In [0]:
df_silver.select(
    [
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df_bronze.columns
    ]
).show()

In [0]:
df_silver.select("category").distinct().show()

In [0]:
df_silver = (
    df_silver.withColumn(
        "division",
        F.when(F.col("category") == "Energy Bars", "Nutrition Bars")
        .when(F.col("category") == "Protein Bars", "Nutrition Bars")
        .when(F.col("category") == "Granola & Cereals", "Breakfast Food")
        .when(F.col("category") == "Recovery Dairy", "Diary & Recovery")
        .when(F.col("category") == "Healthy Snacks", "Healthy Snacks")
        .when(F.col("category") == "Electrolyte Mix", "Hydration & Electrolytes")
    )
)

df_silver.select("division").distinct().show(truncate=False)

In [0]:
df_silver.select("division").distinct().show(truncate=False)

In [0]:
df_silver = df_silver.select(
    F.col("product_id").alias("product_code"),
    "product_name",
    "category",
    "variant",
    "division",
    "read_timestamp",
    "file_name",
    "file_size"
)
df_silver.display(2)

In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
%sql 
SELECT * FROM fmcg.silver.products LIMIT 5

In [0]:
df_gold = df_silver.select("product_code","product_name","category","division","variant")
df_gold.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
%sql
SELECT * FROM fmcg.gold.sb_dim_products LIMIT 5     

In [0]:
delta_table = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.dim_{data_source}")
df_child_products = spark.sql(f"SELECT * FROM {catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
delta_table.alias("target").merge(
    source = df_child_products.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set= {
            "product": "source.product_name",
            "category": "source.category",
            "division": "source.division",
            "variant": "source.variant"
        }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "product": "source.product_name",
        "category": "source.category",
        "division": "source.division",
        "variant": "source.variant"
    }
).execute()